# Piano Fingering Detection — CV Pipeline v3

**Two groups, two approaches:**

- **Group A (Annotation-Assisted)**: Uses dataset annotations (JSON skeletons, keyboard corners). Small set for fast baseline.
- **Group B (Pure Computer Vision)**: Uses **nothing** from the dataset except the video file and MIDI. Keyboard detected automatically via Canny + Hough. Hand landmarks extracted via MediaPipe. No corner annotations, no skeleton JSON.

After comparing, we try to improve Group B by training a BiLSTM on Group A's labeled data and applying it to Group B.

| Stage | Group A | Group B |
|-------|---------|----------|
| Keyboard | Corner annotations | Canny + Hough (auto) |
| Hand pose | Pre-extracted JSON | MediaPipe on raw video |
| MIDI events | TSV annotations | TSV annotations |
| Filtering | Hampel + SavGol | Hampel + SavGol |
| Assignment | Gaussian model | Gaussian model |

**Contents:**
1. [Setup](#0)
2. [Data & Split](#1)
3. [Keyboard Detection Demo](#2)
4. [Hand Pose Estimation Demo](#3)
5. [Temporal Filtering](#4)
6. [Group A — Annotation Baseline](#5)
7. [Group B — Pure CV Pipeline](#6)
8. [A vs B Comparison](#7)
9. [Improving B with A (BiLSTM)](#8)
10. [Final Results](#9)

---
<a id='0'></a>
## 0. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    REPO_URL = 'https://github.com/esnylmz/computer-vision.git'
    BRANCH = 'besn3'
    if not os.path.exists('computer-vision'):
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL], check=True)
    os.chdir('computer-vision')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe-numpy2'], check=True)
    print('\nColab ready')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    try:
        import mediapipe as _mp
        if not hasattr(_mp, 'solutions'):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe-numpy2'], check=True)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe-numpy2'], check=True)
    print('Local ready')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import json, time, warnings
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_style('whitegrid')

print(f'NumPy  : {np.__version__}')
print(f'OpenCV : {cv2.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

In [ ]:
from src.data.dataset import PianoVAMDataset
from src.data.video_utils import VideoProcessor
from src.utils.config import load_config

from src.keyboard.detector import KeyboardDetector, KeyboardRegion
from src.hand.skeleton_loader import SkeletonLoader
from src.hand.temporal_filter import TemporalFilter
from src.hand.fingertip_extractor import FingertipExtractor

from src.assignment.gaussian_assignment import GaussianFingerAssigner, FingerAssignment
from src.assignment.midi_sync import MidiVideoSync

from src.refinement.model import FingeringRefiner, FeatureExtractor, SequenceDataset
from src.refinement.train import train_refiner
from src.refinement.decoding import constrained_viterbi_decode
from src.refinement.constraints import BiomechanicalConstraints

from src.evaluation.metrics import FingeringMetrics

try:
    config = load_config('configs/colab.yaml' if IN_COLAB else 'configs/default.yaml')
except FileNotFoundError:
    config = load_config('configs/default.yaml')

print(f'Config: fps={config.video_fps}, sigma={config.assignment.sigma}')

---
<a id='1'></a>
## 1. Data & Split

In [ ]:
train_dataset = PianoVAMDataset(split='train', max_samples=20)

# Group A: small, fast baseline (increase later if needed)
# Group B: pure CV samples
NUM_GROUP_A = 3
NUM_GROUP_B = 3
MAX_DURATION_SEC = 120
FRAME_SKIP = 2
FRAME_W, FRAME_H = 1920, 1080

all_samples = [train_dataset[i] for i in range(len(train_dataset))]
group_a_samples = all_samples[:NUM_GROUP_A]
group_b_samples = all_samples[NUM_GROUP_A:NUM_GROUP_A + NUM_GROUP_B]

print(f'Group A (annotations): {len(group_a_samples)} samples')
for s in group_a_samples:
    print(f'  {s.id} - {s.metadata["composer"]}')

print(f'\nGroup B (pure CV): {len(group_b_samples)} samples')
for s in group_b_samples:
    print(f'  {s.id} - {s.metadata["composer"]}')

---
<a id='2'></a>
## 2. Keyboard Detection Demo

Classical CV pipeline: grayscale -> blur -> Canny -> Hough lines -> bounding box -> homography -> 88 keys.

In [ ]:
demo_sample = group_b_samples[0]
print(f'Demo sample: {demo_sample.id}')

video_path = train_dataset.download_file(demo_sample.video_path)

vp = VideoProcessor()
vp.open(video_path)
print(f'Resolution: {vp.info.width}x{vp.info.height}, FPS: {vp.info.fps}, Duration: {vp.info.duration:.1f}s')

mid_idx = vp.info.frame_count // 2
frame_bgr = vp.get_frame(mid_idx)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
vp.close()

plt.figure(figsize=(14, 6))
plt.imshow(frame_rgb)
plt.title(f'Raw Video Frame ({mid_idx})')
plt.axis('off')
plt.show()

In [ ]:
# Canny edge detection with different thresholds
gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

edges_low = cv2.Canny(blurred, 30, 100)
edges_mid = cv2.Canny(blurred, 50, 150)
edges_high = cv2.Canny(blurred, 100, 200)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0, 0].imshow(frame_rgb)
axes[0, 0].set_title('Original')
axes[0, 1].imshow(gray, cmap='gray')
axes[0, 1].set_title('Grayscale')
axes[0, 2].imshow(blurred, cmap='gray')
axes[0, 2].set_title('Gaussian Blur')
axes[1, 0].imshow(edges_low, cmap='gray')
axes[1, 0].set_title('Canny (30, 100)')
axes[1, 1].imshow(edges_mid, cmap='gray')
axes[1, 1].set_title('Canny (50, 150)')
axes[1, 2].imshow(edges_high, cmap='gray')
axes[1, 2].set_title('Canny (100, 200)')
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Edge Detection Pipeline', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Hough Line Transform
edges = edges_mid
lines = cv2.HoughLinesP(edges, rho=1, theta=np.pi/180, threshold=100,
                         minLineLength=100, maxLineGap=10)

line_vis = frame_rgb.copy()
h_lines, v_lines = [], []
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.abs(np.arctan2(y2 - y1, x2 - x1))
        if angle < np.pi / 18:
            h_lines.append((x1, y1, x2, y2))
            cv2.line(line_vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        elif angle > np.pi / 2 - np.pi / 18:
            v_lines.append((x1, y1, x2, y2))
            cv2.line(line_vis, (x1, y1), (x2, y2), (255, 0, 0), 1)

print(f'Lines: {len(lines) if lines is not None else 0} total, {len(h_lines)} horizontal, {len(v_lines)} vertical')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].imshow(edges, cmap='gray')
axes[0].set_title('Canny Edges')
axes[1].imshow(line_vis)
axes[1].set_title('Hough Lines')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# automatic keyboard detection (no corners) with multiple parameter sets
def auto_detect_keyboard(frame_bgr, verbose=True):
    """Try multiple Canny/Hough parameter combos. Return best KeyboardRegion or None."""
    param_sets = [
        {'canny_low': 50, 'canny_high': 150, 'hough_threshold': 100, 'min_line_length': 100},
        {'canny_low': 30, 'canny_high': 100, 'hough_threshold': 80,  'min_line_length': 80},
        {'canny_low': 40, 'canny_high': 120, 'hough_threshold': 60,  'min_line_length': 60},
        {'canny_low': 60, 'canny_high': 180, 'hough_threshold': 120, 'min_line_length': 120},
        {'canny_low': 20, 'canny_high': 80,  'hough_threshold': 50,  'min_line_length': 50},
    ]

    best_region = None
    best_keys = 0

    for params in param_sets:
        det = KeyboardDetector(params)
        region = det.detect(frame_bgr)
        if region and len(region.key_boundaries) > best_keys:
            best_region = region
            best_keys = len(region.key_boundaries)
            if verbose:
                print(f'    params {params} -> {best_keys} keys')

    return best_region


def auto_detect_keyboard_from_video(video_path, max_attempts=10, verbose=True):
    """Try auto detection on multiple frames with multiple parameter sets."""
    vp = VideoProcessor()
    vp.open(video_path)
    total = vp.info.frame_count

    # try frames spread across the video
    trial_indices = [int(total * f) for f in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.15, 0.45]]
    trial_indices = trial_indices[:max_attempts]

    best_region = None
    best_keys = 0
    best_frame_idx = None

    for fidx in trial_indices:
        frame = vp.get_frame(fidx)
        if frame is None:
            continue
        if verbose:
            print(f'  Frame {fidx}:')
        region = auto_detect_keyboard(frame, verbose=verbose)
        if region and len(region.key_boundaries) > best_keys:
            best_region = region
            best_keys = len(region.key_boundaries)
            best_frame_idx = fidx

    vp.close()

    if verbose:
        if best_region:
            print(f'  Best: {best_keys} keys from frame {best_frame_idx}')
        else:
            print(f'  FAILED: no keyboard detected on any frame')

    return best_region, best_frame_idx


# test on demo video
print(f'Auto-detecting keyboard from video {demo_sample.id}...')
demo_kb, demo_kb_frame = auto_detect_keyboard_from_video(video_path)

if demo_kb:
    print(f'\nSuccess: {len(demo_kb.key_boundaries)} keys, bbox={demo_kb.bbox}')
else:
    print('\nFailed on all frames and parameter sets')

In [ ]:
# visualize auto-detected keyboard (if found)
if demo_kb:
    vp = VideoProcessor()
    vp.open(video_path)
    kb_frame = vp.get_frame(demo_kb_frame)
    vp.close()

    kb_vis = cv2.cvtColor(kb_frame, cv2.COLOR_BGR2RGB).copy()
    x1, y1, x2, y2 = demo_kb.bbox
    cv2.rectangle(kb_vis, (x1, y1), (x2, y2), (0, 255, 0), 3)

    # draw some key boundaries
    for ki, (kx1, ky1, kx2, ky2) in demo_kb.key_boundaries.items():
        midi = ki + 21
        is_black = (midi % 12) in [1, 3, 6, 8, 10]
        color = (150, 150, 150) if is_black else (200, 200, 200)
        cv2.rectangle(kb_vis, (kx1, ky1), (kx2, ky2), color, 1)

    # warp
    H = demo_kb.homography
    w_size = (max(int(demo_kb.bbox[2] - demo_kb.bbox[0]), 800), max(int(demo_kb.bbox[3] - demo_kb.bbox[1]), 100))
    warped = cv2.warpPerspective(kb_frame, H, w_size)
    warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    axes[0].imshow(kb_vis)
    axes[0].set_title(f'Auto-Detected Keyboard (frame {demo_kb_frame})')
    axes[1].imshow(warped_rgb)
    axes[1].set_title('Perspective-Corrected')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No keyboard detected - skipping visualization')

---
<a id='3'></a>
## 3. Hand Pose Estimation Demo

MediaPipe Hands on raw video frames - 21 landmarks per hand.

In [ ]:
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
print(f'MediaPipe: {mp.__version__}')

In [ ]:
# detect hands on sample frames
vp = VideoProcessor()
vp.open(video_path)

test_indices = [300, 900, 1800, 3600]
test_frames = []
for idx in test_indices:
    f = vp.get_frame(idx)
    if f is not None:
        test_frames.append((idx, f))
vp.close()

det = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)

fig, axes = plt.subplots(1, len(test_frames), figsize=(5 * len(test_frames), 6))
if len(test_frames) == 1:
    axes = [axes]

for i, (fidx, frame) in enumerate(test_frames):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = det.process(rgb)
    annotated = rgb.copy()
    n = 0
    if res.multi_hand_landmarks:
        n = len(res.multi_hand_landmarks)
        for hlm in res.multi_hand_landmarks:
            mp_drawing.draw_landmarks(annotated, hlm, mp_hands.HAND_CONNECTIONS,
                                      mp_drawing_styles.get_default_hand_landmarks_style(),
                                      mp_drawing_styles.get_default_hand_connections_style())
    axes[i].imshow(annotated)
    axes[i].set_title(f'Frame {fidx} ({n} hands)')
    axes[i].axis('off')

det.close()
plt.suptitle('MediaPipe Hand Detection', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# fingertip labels on one frame
demo_bgr = test_frames[1][1] if len(test_frames) > 1 else test_frames[0][1]
demo_rgb = cv2.cvtColor(demo_bgr, cv2.COLOR_BGR2RGB)

det = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)
res = det.process(demo_rgb)
det.close()

h, w = demo_rgb.shape[:2]
annotated = demo_rgb.copy()
tip_ids = [4, 8, 12, 16, 20]
tip_names = {4: 'Thumb', 8: 'Index', 12: 'Middle', 16: 'Ring', 20: 'Pinky'}

if res.multi_hand_landmarks:
    for hlm, hinfo in zip(res.multi_hand_landmarks, res.multi_handedness):
        label = hinfo.classification[0].label
        mp_drawing.draw_landmarks(annotated, hlm, mp_hands.HAND_CONNECTIONS)
        for tid in tip_ids:
            lm = hlm.landmark[tid]
            px, py = int(lm.x * w), int(lm.y * h)
            cv2.circle(annotated, (px, py), 8, (255, 0, 0), -1)
            cv2.putText(annotated, tip_names[tid], (px + 10, py - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        print(f'{label} hand:')
        for tid in tip_ids:
            lm = hlm.landmark[tid]
            print(f'  {tip_names[tid]:6s}: ({lm.x:.4f}, {lm.y:.4f}, {lm.z:.4f})')

plt.figure(figsize=(14, 8))
plt.imshow(annotated)
plt.title('21-Keypoint Skeleton with Fingertip Labels')
plt.axis('off')
plt.show()

---
<a id='4'></a>
## 4. Temporal Filtering

Raw MediaPipe output has noise and missing frames. We apply Hampel (outlier removal) + interpolation + Savitzky-Golay (smoothing).

In [ ]:
# quick demo: extract landmarks from ~200 frames, show raw vs filtered
vp = VideoProcessor()
vp.open(video_path)

n_demo = 400
raw_lms = np.full((n_demo, 21, 3), np.nan)

det = mp_hands.Hands(static_image_mode=False, max_num_hands=2,
                      min_detection_confidence=0.5, min_tracking_confidence=0.4)

for fidx in range(0, n_demo, 2):
    frame = vp.get_frame(fidx + 300)
    if frame is None:
        continue
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    r = det.process(rgb)
    if r.multi_hand_landmarks:
        for hlm, hinfo in zip(r.multi_hand_landmarks, r.multi_handedness):
            if hinfo.classification[0].label.lower() == 'right':
                raw_lms[fidx] = np.array([[l.x, l.y, l.z] for l in hlm.landmark])

det.close()
vp.close()

tf = TemporalFilter(
    hampel_window=config.hand.hampel_window,
    hampel_threshold=config.hand.hampel_threshold,
    max_interpolation_gap=config.hand.interpolation_max_gap,
    savgol_window=config.hand.savgol_window,
    savgol_order=config.hand.savgol_order
)
filtered_lms = tf.process(raw_lms.copy())

valid_raw = int(np.sum(~np.all(np.isnan(raw_lms.reshape(n_demo, -1)), axis=1)))
print(f'Detected hand in {valid_raw}/{n_demo} frames')

# plot index fingertip trajectory
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
axes[0].plot(raw_lms[:, 8, 0], alpha=0.4, label='Raw', marker='.', markersize=2, linewidth=0)
axes[0].plot(filtered_lms[:, 8, 0], label='Filtered', linewidth=1.5)
axes[0].set_ylabel('x (norm)')
axes[0].set_title('Index fingertip - raw vs filtered')
axes[0].legend()

axes[1].plot(raw_lms[:, 8, 1], alpha=0.4, label='Raw', marker='.', markersize=2, linewidth=0)
axes[1].plot(filtered_lms[:, 8, 1], label='Filtered', linewidth=1.5)
axes[1].set_ylabel('y (norm)')
axes[1].set_xlabel('Frame')
axes[1].legend()
plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. Group A — Annotation Baseline

Uses pre-extracted JSON skeletons + keyboard corner annotations. Fast and reliable.

In [ ]:
def project_keys_to_pixel_space(key_boundaries_warped, homography):
    H_inv = np.linalg.inv(homography)
    result = {}
    for k, (x1, y1, x2, y2) in key_boundaries_warped.items():
        cy = (y1 + y2) / 2.0
        pts_w = np.array([[x1, cy, 1.0], [x2, cy, 1.0],
                          [(x1 + x2) / 2.0, cy, 1.0]], dtype=np.float64)
        pts_p = (H_inv @ pts_w.T).T
        pts_p = pts_p[:, :2] / pts_p[:, 2:3]
        lx, rx = pts_p[0, 0], pts_p[1, 0]
        cy_px = pts_p[2, 1]
        result[k] = (lx, cy_px - 5.0, rx, cy_px + 5.0)
    return result


def run_assignment(keys_px, left_px, right_px, midi_events, fps):
    sync = MidiVideoSync(fps=fps)
    synced = sync.sync_events(midi_events)

    assigner = GaussianFingerAssigner(
        keys_px, sigma=config.assignment.sigma,
        candidate_range=config.assignment.candidate_keys)

    assignments = []
    for event in synced:
        fi, ki = event.frame_idx, event.key_idx
        if ki not in assigner.key_centers:
            continue

        asgn_r = None
        if fi < len(right_px):
            lm = right_px[fi]
            if not np.any(np.isnan(lm)):
                asgn_r = assigner.assign_from_landmarks(lm, ki, 'right', fi, event.onset_time)

        asgn_l = None
        if fi < len(left_px):
            lm = left_px[fi]
            if not np.any(np.isnan(lm)):
                asgn_l = assigner.assign_from_landmarks(lm, ki, 'left', fi, event.onset_time)

        cands = [a for a in (asgn_r, asgn_l) if a is not None]
        if cands:
            assignments.append(max(cands, key=lambda a: a.confidence))

    return assignments


def load_midi_events(sample, dataset, max_duration):
    tsv = dataset.load_tsv_annotations(sample)
    events = []
    for _, row in tsv.iterrows():
        t = float(row['onset'])
        if t > max_duration:
            continue
        events.append({'onset': t, 'offset': t + 0.2,
                       'pitch': int(row['note']),
                       'velocity': int(row.get('velocity', 64))})
    return events


def process_sample_A(sample, dataset):
    """Group A: use annotations."""
    result = {'sample_id': sample.id, 'assignments': [], 'error': None}
    try:
        # keyboard from corners
        kb = KeyboardDetector().detect_from_corners(sample.metadata['keyboard_corners'])
        keys_px = project_keys_to_pixel_space(kb.key_boundaries, kb.homography)

        # skeleton from JSON
        skel = dataset.load_skeleton(sample)
        loader = SkeletonLoader()
        hands = loader._parse_json(skel)
        left_arr = loader.to_array(hands['left'])
        right_arr = loader.to_array(hands['right'])

        # filter
        filt = TemporalFilter(
            hampel_window=config.hand.hampel_window,
            hampel_threshold=config.hand.hampel_threshold,
            max_interpolation_gap=config.hand.interpolation_max_gap,
            savgol_window=config.hand.savgol_window,
            savgol_order=config.hand.savgol_order)
        lf = filt.process(left_arr.copy()) if left_arr.size > 0 else left_arr
        rf = filt.process(right_arr.copy()) if right_arr.size > 0 else right_arr

        # scale to pixels
        left_px = lf.copy()
        right_px = rf.copy()
        if left_px.size > 0:
            left_px[:, :, 0] *= FRAME_W
            left_px[:, :, 1] *= FRAME_H
        if right_px.size > 0:
            right_px[:, :, 0] *= FRAME_W
            right_px[:, :, 1] *= FRAME_H

        # limit duration
        max_f = int(MAX_DURATION_SEC * config.video_fps)
        if left_px.size > 0 and len(left_px) > max_f:
            left_px = left_px[:max_f]
        if right_px.size > 0 and len(right_px) > max_f:
            right_px = right_px[:max_f]

        midi_events = load_midi_events(sample, dataset, MAX_DURATION_SEC)
        result['assignments'] = run_assignment(keys_px, left_px, right_px, midi_events, config.video_fps)
    except Exception as e:
        result['error'] = str(e)
    return result


print('Group A functions defined')

In [ ]:
print(f'Group A: {len(group_a_samples)} samples\n')
group_a_results = []

for i, s in enumerate(group_a_samples):
    print(f'[{i+1}/{len(group_a_samples)}] {s.id}...', end=' ')
    t0 = time.time()
    res = process_sample_A(s, train_dataset)
    dt = time.time() - t0
    if res['error']:
        print(f'ERROR: {res["error"]}')
    else:
        print(f'{len(res["assignments"])} assignments ({dt:.1f}s)')
    group_a_results.append(res)

total_a = sum(len(r['assignments']) for r in group_a_results)
print(f'\nGroup A: {total_a} total assignments')

---
<a id='6'></a>
## 6. Group B — Pure CV Pipeline

**No annotations used.** Keyboard detected automatically from video (Canny + Hough). Hand landmarks extracted with MediaPipe. If automatic keyboard detection fails, the sample is **skipped** (no corner fallback).

In [ ]:
def extract_landmarks_mp(video_path, max_frames, frame_skip=2):
    """Run MediaPipe on video, return (T, 21, 3) arrays for left and right."""
    vp = VideoProcessor()
    vp.open(video_path)
    total = min(max_frames, vp.info.frame_count)

    left_lms = np.full((total, 21, 3), np.nan)
    right_lms = np.full((total, 21, 3), np.nan)

    detector = mp_hands.Hands(
        static_image_mode=False, max_num_hands=2,
        min_detection_confidence=0.5, min_tracking_confidence=0.4)

    for fidx in tqdm(range(0, total, frame_skip), desc='MediaPipe', leave=False):
        frame = vp.get_frame(fidx)
        if frame is None:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detector.process(rgb)
        if results.multi_hand_landmarks:
            for hlm, hinfo in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = hinfo.classification[0].label.lower()
                arr = np.array([[l.x, l.y, l.z] for l in hlm.landmark])
                if label == 'left':
                    left_lms[fidx] = arr
                else:
                    right_lms[fidx] = arr

    detector.close()
    vp.close()

    # interpolate skipped frames
    for lms in [left_lms, right_lms]:
        for idx in range(total):
            if idx % frame_skip != 0 and np.all(np.isnan(lms[idx])):
                prev = idx - (idx % frame_skip)
                nxt = prev + frame_skip
                if nxt < total and not np.all(np.isnan(lms[prev])) and not np.all(np.isnan(lms[nxt])):
                    alpha = (idx - prev) / frame_skip
                    lms[idx] = (1 - alpha) * lms[prev] + alpha * lms[nxt]

    left_v = int(np.sum(~np.all(np.isnan(left_lms.reshape(total, -1)), axis=1)))
    right_v = int(np.sum(~np.all(np.isnan(right_lms.reshape(total, -1)), axis=1)))
    return left_lms, right_lms, left_v, right_v


def process_sample_B(sample, dataset):
    """Group B: pure CV. No annotations. Skip if keyboard auto-detect fails."""
    result = {'sample_id': sample.id, 'assignments': [], 'error': None,
              'kb_method': None, 'skipped': False}
    try:
        vid_path = dataset.download_file(sample.video_path)
        vp = VideoProcessor()
        vp.open(vid_path)
        fps = vp.info.fps or config.video_fps
        max_frames = int(MAX_DURATION_SEC * fps)
        vp.close()

        # keyboard: automatic only, NO fallback
        print('  Detecting keyboard (auto)...')
        kb, kb_fidx = auto_detect_keyboard_from_video(vid_path, max_attempts=10, verbose=False)

        if kb is None or len(kb.key_boundaries) < 50:
            print('  Keyboard detection FAILED - skipping sample')
            result['skipped'] = True
            result['kb_method'] = 'FAILED'
            return result

        result['kb_method'] = f'auto (frame {kb_fidx}, {len(kb.key_boundaries)} keys)'
        print(f'  Keyboard: {result["kb_method"]}')

        keys_px = project_keys_to_pixel_space(kb.key_boundaries, kb.homography)

        # hand landmarks: MediaPipe only
        print('  Extracting hand landmarks (MediaPipe)...')
        left_cv, right_cv, left_v, right_v = extract_landmarks_mp(vid_path, max_frames, FRAME_SKIP)
        print(f'  Valid frames - L: {left_v}, R: {right_v}')

        # temporal filtering
        filt = TemporalFilter(
            hampel_window=config.hand.hampel_window,
            hampel_threshold=config.hand.hampel_threshold,
            max_interpolation_gap=config.hand.interpolation_max_gap,
            savgol_window=config.hand.savgol_window,
            savgol_order=config.hand.savgol_order)
        left_f = filt.process(left_cv.copy()) if left_cv.size > 0 else left_cv
        right_f = filt.process(right_cv.copy()) if right_cv.size > 0 else right_cv

        # scale to pixels
        left_px = left_f.copy()
        right_px = right_f.copy()
        if left_px.size > 0:
            left_px[:, :, 0] *= FRAME_W
            left_px[:, :, 1] *= FRAME_H
        if right_px.size > 0:
            right_px[:, :, 0] *= FRAME_W
            right_px[:, :, 1] *= FRAME_H

        # MIDI events (this is the task input, not a CV annotation)
        midi_events = load_midi_events(sample, dataset, MAX_DURATION_SEC)

        # assignment
        result['assignments'] = run_assignment(keys_px, left_px, right_px, midi_events, fps)
        print(f'  Assignments: {len(result["assignments"])}')

    except Exception as e:
        result['error'] = str(e)
    return result


print('Group B functions defined')

In [ ]:
print(f'Group B: {len(group_b_samples)} samples (pure CV)\n')

group_b_results = []
for i, s in enumerate(group_b_samples):
    print(f'[{i+1}/{len(group_b_samples)}] {s.id}:')
    t0 = time.time()
    res = process_sample_B(s, train_dataset)
    dt = time.time() - t0
    if res['error']:
        print(f'  ERROR: {res["error"]}')
    elif not res['skipped']:
        print(f'  Done ({dt:.1f}s)')
    group_b_results.append(res)
    print()

b_ok = [r for r in group_b_results if not r.get('skipped') and not r.get('error')]
b_skip = [r for r in group_b_results if r.get('skipped')]
print(f'Group B: {len(b_ok)} succeeded, {len(b_skip)} skipped (keyboard detection failed)')
print(f'Total assignments: {sum(len(r["assignments"]) for r in b_ok)}')

---
<a id='7'></a>
## 7. A vs B Comparison

Compare annotation-based results (A) against pure CV results (B).

In [ ]:
bc = BiomechanicalConstraints()

def get_stats(results, name):
    rows = []
    for res in results:
        asgns = res['assignments']
        if len(asgns) < 2:
            continue
        fingers = [a.assigned_finger for a in asgns]
        pitches = [a.midi_pitch for a in asgns]
        hands = [a.hand for a in asgns]
        confs = [a.confidence for a in asgns]
        viols = bc.validate_sequence(fingers, pitches, hands)
        ifr = len(viols) / max(len(fingers) - 1, 1)
        rows.append({
            'group': name, 'sample': res['sample_id'],
            'n': len(asgns), 'ifr': ifr,
            'conf': np.mean(confs),
            'right_pct': sum(1 for a in asgns if a.hand == 'right') / len(asgns) * 100
        })
    return rows

stats_a = get_stats(group_a_results, 'A')
stats_b = get_stats(b_ok, 'B')
all_stats = stats_a + stats_b

print(f'{"Group":<8} {"Sample":<20} {"N":>5} {"IFR":>8} {"Conf":>8} {"R%":>6}')
print('-' * 58)
for st in all_stats:
    print(f'{st["group"]:<8} {st["sample"]:<20} {st["n"]:>5} {st["ifr"]:>8.3f} {st["conf"]:>8.3f} {st["right_pct"]:>5.1f}%')

if stats_a:
    print(f'\nA mean: IFR={np.mean([s["ifr"] for s in stats_a]):.4f}, Conf={np.mean([s["conf"] for s in stats_a]):.4f}')
if stats_b:
    print(f'B mean: IFR={np.mean([s["ifr"] for s in stats_b]):.4f}, Conf={np.mean([s["conf"] for s in stats_b]):.4f}')

In [ ]:
# bar charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels = ['A (Annotations)', 'B (Pure CV)']
colors = ['steelblue', 'darkorange']

if stats_a and stats_b:
    axes[0].bar(labels, [np.mean([s['ifr'] for s in stats_a]),
                          np.mean([s['ifr'] for s in stats_b])], color=colors)
    axes[0].set_title('IFR (lower = better)')

    axes[1].bar(labels, [np.mean([s['conf'] for s in stats_a]),
                          np.mean([s['conf'] for s in stats_b])], color=colors)
    axes[1].set_title('Confidence (higher = better)')

    axes[2].bar(labels, [np.mean([s['n'] for s in stats_a]),
                          np.mean([s['n'] for s in stats_b])], color=colors)
    axes[2].set_title('Assignments per sample')
elif stats_a:
    axes[0].bar(['A'], [np.mean([s['ifr'] for s in stats_a])], color='steelblue')
    axes[0].set_title('IFR')
    axes[1].text(0.5, 0.5, 'Group B: no\nsuccessful samples', ha='center', va='center', transform=axes[1].transAxes)

plt.suptitle('Group A vs Group B', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# confidence distributions
fig, ax = plt.subplots(figsize=(10, 5))

confs_a = [a.confidence for r in group_a_results for a in r['assignments']]
confs_b = [a.confidence for r in b_ok for a in r['assignments']]

if confs_a:
    ax.hist(confs_a, bins=40, alpha=0.6, color='steelblue', label=f'A (n={len(confs_a)})')
if confs_b:
    ax.hist(confs_b, bins=40, alpha=0.6, color='darkorange', label=f'B (n={len(confs_b)})')

ax.set_title('Confidence Distribution')
ax.set_xlabel('Confidence')
ax.legend()
plt.tight_layout()
plt.show()

---
<a id='8'></a>
## 8. Improving B with A (BiLSTM)

Train a BiLSTM on Group A's labeled assignments, then apply it to refine Group B. This tests whether knowledge from annotated data can improve pure CV results.

In [ ]:
# build training sequences from Group A
train_sequences = []
for res in group_a_results:
    asgns = res['assignments']
    if len(asgns) < 10:
        continue
    train_sequences.append({
        'pitches': [a.midi_pitch for a in asgns],
        'fingers': [a.assigned_finger for a in asgns],
        'onsets': [a.note_onset for a in asgns],
        'hands': [a.hand for a in asgns],
        'labels': [a.assigned_finger for a in asgns],
    })

print(f'Training sequences: {len(train_sequences)}')
if not train_sequences:
    print('Not enough data to train - skipping BiLSTM')

In [ ]:
trained_model = None

if train_sequences:
    feature_extractor = FeatureExtractor(normalize_pitch=True)

    split_idx = max(1, int(len(train_sequences) * 0.8))
    train_ds = SequenceDataset(train_sequences[:split_idx], feature_extractor, max_len=256)
    val_ds = SequenceDataset(train_sequences[split_idx:], feature_extractor, max_len=256) if split_idx < len(train_sequences) else None

    t_config = {
        'hidden_size': config.refinement.hidden_size,
        'num_layers': config.refinement.num_layers,
        'dropout': config.refinement.dropout,
        'batch_size': min(config.refinement.batch_size, len(train_ds)),
        'learning_rate': config.refinement.learning_rate,
        'epochs': min(20, config.refinement.epochs),
        'early_stopping_patience': 5,
        'device': DEVICE,
        'checkpoint_dir': '/content/checkpoints' if IN_COLAB else './checkpoints'
    }

    print('Training BiLSTM...')
    trained_model = train_refiner(train_ds, val_ds, t_config)
    print('Done.')
else:
    print('Skipped (no training data)')

In [ ]:
def refine(assignments, model, feat_ext, device=DEVICE):
    if model is None or len(assignments) < 5:
        return assignments

    pitches = [a.midi_pitch for a in assignments]
    fingers = [a.assigned_finger for a in assignments]
    onsets = [a.note_onset for a in assignments]
    hands_list = [a.hand for a in assignments]

    features = feat_ext.extract(pitches, fingers, onsets, hands_list).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(features)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0][:len(assignments)]

    decoded = constrained_viterbi_decode(probs, pitches, hands_list, BiomechanicalConstraints(strict=False))

    return [FingerAssignment(
        note_onset=a.note_onset, frame_idx=a.frame_idx, midi_pitch=a.midi_pitch,
        key_idx=a.key_idx, assigned_finger=nf, hand=a.hand,
        confidence=a.confidence, fingertip_position=a.fingertip_position
    ) for a, nf in zip(assignments, decoded.fingers)]


# refine both groups
if trained_model:
    print('Refining Group A...')
    a_refined = [{'sample_id': r['sample_id'],
                  'assignments': refine(r['assignments'], trained_model, feature_extractor)}
                 for r in group_a_results]
    print(f'  {sum(len(r["assignments"]) for r in a_refined)} assignments')

    print('Refining Group B...')
    b_refined = [{'sample_id': r['sample_id'],
                  'assignments': refine(r['assignments'], trained_model, feature_extractor)}
                 for r in b_ok]
    print(f'  {sum(len(r["assignments"]) for r in b_refined)} assignments')
else:
    a_refined = group_a_results
    b_refined = b_ok
    print('No model - using baseline results as-is')

---
<a id='9'></a>
## 9. Final Results

In [ ]:
def compute_ifrs(results):
    ifrs = []
    for res in results:
        a = res['assignments']
        if len(a) < 2:
            continue
        f = [x.assigned_finger for x in a]
        p = [x.midi_pitch for x in a]
        h = [x.hand for x in a]
        v = bc.validate_sequence(f, p, h)
        ifrs.append(len(v) / max(len(f) - 1, 1))
    return ifrs

ifr_a_base = compute_ifrs(group_a_results)
ifr_a_ref = compute_ifrs(a_refined)
ifr_b_base = compute_ifrs(b_ok)
ifr_b_ref = compute_ifrs(b_refined)

print(f'{"":<30} {"Baseline":>12} {"Refined":>12} {"Change":>12}')
print('-' * 68)
if ifr_a_base:
    ab, ar = np.mean(ifr_a_base), np.mean(ifr_a_ref)
    print(f'{"A (Annotations)":<30} {ab:>12.4f} {ar:>12.4f} {ar - ab:>+12.4f}')
if ifr_b_base:
    bb, br = np.mean(ifr_b_base), np.mean(ifr_b_ref)
    print(f'{"B (Pure CV)":<30} {bb:>12.4f} {br:>12.4f} {br - bb:>+12.4f}')

print(f'\nKeyboard detection (Group B):')
for r in group_b_results:
    status = r.get('kb_method', 'unknown')
    print(f'  {r["sample_id"]}: {status}')

In [ ]:
# final comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cats = ['A Base', 'A Refined', 'B Base', 'B Refined']
cols = ['steelblue', 'royalblue', 'darkorange', 'orangered']

ifr_vals = [
    np.mean(ifr_a_base) if ifr_a_base else 0,
    np.mean(ifr_a_ref) if ifr_a_ref else 0,
    np.mean(ifr_b_base) if ifr_b_base else 0,
    np.mean(ifr_b_ref) if ifr_b_ref else 0
]
axes[0].bar(cats, ifr_vals, color=cols)
axes[0].set_title('IFR (lower = better)')

conf_vals = []
for results in [group_a_results, a_refined, b_ok, b_refined]:
    c = [a.confidence for r in results for a in r['assignments']]
    conf_vals.append(np.mean(c) if c else 0)
axes[1].bar(cats, conf_vals, color=cols)
axes[1].set_title('Confidence (higher = better)')

plt.suptitle('Annotations vs Pure CV — Baseline vs Refined', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# finger distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fnames = {1: 'Thumb', 2: 'Index', 3: 'Middle', 4: 'Ring', 5: 'Pinky'}

for ax, results, title in [
    (axes[0, 0], group_a_results, 'A Baseline'),
    (axes[0, 1], a_refined, 'A Refined'),
    (axes[1, 0], b_ok, 'B Baseline'),
    (axes[1, 1], b_refined, 'B Refined'),
]:
    fingers = [a.assigned_finger for r in results for a in r['assignments']]
    if fingers:
        counts = pd.Series(fingers).value_counts().sort_index()
        counts.index = [fnames.get(i, str(i)) for i in counts.index]
        counts.plot.bar(ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_ylabel('Count')

plt.suptitle('Finger Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# save
save_dir = Path('/content/results' if IN_COLAB else './results')
save_dir.mkdir(parents=True, exist_ok=True)

summary = {
    'group_a': {
        'n_samples': len(group_a_results),
        'total_assignments': sum(len(r['assignments']) for r in group_a_results),
        'ifr_baseline': float(np.mean(ifr_a_base)) if ifr_a_base else None,
        'ifr_refined': float(np.mean(ifr_a_ref)) if ifr_a_ref else None,
    },
    'group_b': {
        'n_samples': len(group_b_samples),
        'n_succeeded': len(b_ok),
        'n_skipped': len(b_skip),
        'total_assignments': sum(len(r['assignments']) for r in b_ok),
        'ifr_baseline': float(np.mean(ifr_b_base)) if ifr_b_base else None,
        'ifr_refined': float(np.mean(ifr_b_ref)) if ifr_b_ref else None,
        'keyboard_methods': {r['sample_id']: r.get('kb_method', '') for r in group_b_results},
    },
    'config': {
        'max_duration_sec': MAX_DURATION_SEC,
        'frame_skip': FRAME_SKIP,
        'num_group_a': NUM_GROUP_A,
        'num_group_b': NUM_GROUP_B
    }
}

with open(save_dir / 'results_v3.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

if trained_model:
    torch.save(trained_model.state_dict(), save_dir / 'bilstm_v3.pt')

print(f'Saved to {save_dir}')
print(json.dumps(summary, indent=2, default=str))